# Bengali Handwritten OCR Fine-Tuning with Qwen2-VL (QLoRA) on Kaggle
Train on the BN-HTRd dataset using 4-bit NF4 QLoRA on Kaggle Dual Tesla T4 GPUs (30 hours/week free).

> **Important Settings** in the right-hand panel:
- **Accelerator**: `GPU T4 x2`
- **Internet**: `Internet on`
- **Persistence**: `Variables and Files`

In [ ]:
# 1. Verify Dual GPU availability
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU found! Enable GPU T4 x2 in Session Options."
print(f"CUDA Device count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# 2. Install required packages
!pip install -q "transformers>=4.45.0" "accelerate>=0.30.0" "peft>=0.11.0" "bitsandbytes>=0.43.0" "qwen-vl-utils>=0.0.8" datasets pandas openpyxl pillow jiwer gdown

In [ ]:
# 3. Setup workspace & clone repository
import os
%cd /kaggle/working
if not os.path.exists('/kaggle/working/bangla-ocr-qwen2vl'):
    !git clone https://github.com/RahulIslam46/bangla-ocr-qwen2vl.git /kaggle/working/bangla-ocr-qwen2vl
%cd /kaggle/working/bangla-ocr-qwen2vl
!git pull
!mkdir -p /kaggle/working/bangla_ocr_checkpoints

In [ ]:
# 4. Download and Extract BN-HTRd Benchmark Dataset (~1.38 GB)
import os, glob, zipfile

data_dir = "/kaggle/working/data/Dataset"
zip_path = "/kaggle/working/BN-HTRd.zip"

if not os.path.exists(data_dir) or len(glob.glob(f"{data_dir}/*")) == 0:
    if not os.path.exists(zip_path):
        print("Downloading BN-HTRd dataset (~1.38 GB) from Google Drive...")
        !gdown --id 16eenWCoi4MahIJmIJ4iYPAGK9aL8X1Xy -O /kaggle/working/BN-HTRd.zip
    
    print("Extracting outer BN-HTRd archive...")
    !mkdir -p /kaggle/working/data
    !unzip -q -o /kaggle/working/BN-HTRd.zip -d /kaggle/working/data/
    
    print("Extracting inner Dataset.zip...")
    nested = glob.glob("/kaggle/working/data/**/Dataset.zip", recursive=True)
    if nested:
        !unzip -q -o "{nested[0]}" -d /kaggle/working/data/
        print("Dataset.zip extracted successfully!")
    else:
        print("Dataset.zip not found in extracted files!")

print(f"Target dataset directory: {data_dir}")
print(f"Directory exists: {os.path.exists(data_dir)}")
if os.path.exists(data_dir):
    folders = [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]
    print(f"Total document folders found: {len(folders)}")

In [ ]:
# 5. Prepare checkpoint for resuming
import os, glob, shutil

candidates = glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
cp_path = None
if candidates:
    source_dir = os.path.dirname(candidates[0])
    cp_name = os.path.basename(source_dir)
    if "checkpoint" not in cp_name:
        cp_name = "checkpoint-1123"
    target_cp_dir = os.path.join("/kaggle/working/bangla_ocr_checkpoints", cp_name)
    os.makedirs(target_cp_dir, exist_ok=True)
    print(f"Found checkpoint source at: {source_dir}")
    for item in os.listdir(source_dir):
        src_path = os.path.join(source_dir, item)
        dst_path = os.path.join(target_cp_dir, item)
        if os.path.isfile(src_path):
            shutil.copy2(src_path, dst_path)
    print(f"Checkpoint successfully prepared at: {target_cp_dir}")
    print("Files in checkpoint:", os.listdir(target_cp_dir))
    cp_path = target_cp_dir
else:
    print("Warning: No checkpoint found in /kaggle/input, will train from scratch.")


In [ ]:
# 6. Launch Training (Resumes seamlessly from checkpoint)
import os, subprocess

cmd = [
    "python", "train.py",
    "--data_dir", "/kaggle/working/data/Dataset",
    "--mode", "line",
    "--output_dir", "/kaggle/working/bangla_ocr_checkpoints",
    "--epochs", "3",
    "--batch_size", "1",
    "--grad_accum", "8",
    "--lr", "2e-4",
    "--save_steps", "50",
    "--logging_steps", "5",
    "--max_val_samples", "150",
    "--max_time_hours", "10.5"
]

if cp_path and os.path.exists(os.path.join(cp_path, "adapter_model.safetensors")):
    cmd.extend(["--resume_from_checkpoint", cp_path])
    print(f"Resuming training from {cp_path}...")
else:
    print("Training from scratch...")

subprocess.run(cmd, check=True)


In [ ]:
# 7. Verify saved checkpoints and outputs
import os, glob
print("=== Training Session Finished Cleanly ===")
cp_dir = "/kaggle/working/bangla_ocr_checkpoints"
if os.path.exists(cp_dir):
    print("Saved files in checkpoints directory:")
    for root, dirs, files in os.walk(cp_dir):
        for f in files:
            p = os.path.join(root, f)
            sz = os.path.getsize(p) / (1024 * 1024)
            print(f"  {p} ({sz:.2f} MB)")
print("Ready for Kaggle automatic output commit!")
